# INC-001 Sleep profiling: `sleep.json`

Read-only; proposes, decides nothing. Per-participant files are read one at a time. Only counts and
(participant_id, date) identifiers are printed. Cell ids (`slp-NN`) are referenced from the findings document.
Timestamps in sleep.json are naive local (UTC-05:00 per the dictionary) with a `.000` fraction.

In [1]:
import json, re, time, collections, statistics
from datetime import date, datetime, timedelta
from pathlib import Path

T0 = time.perf_counter()
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "participants").is_dir())
EVENTS = ROOT / "data" / "wearable_events"
WIN_START, WIN_END = date(2026, 1, 8), date(2026, 2, 4)
WINDOW_DAYS = {WIN_START + timedelta(days=i) for i in range(28)}
TS_RE = re.compile(r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\.000$")
STAGE_ORDER = ["light", "deep", "rem", "light", "awake"]

def show(label, value, examples=None):
    extra = f"  examples={sorted(examples)[:5]}" if examples else ""
    print(f"{label}: {value}{extra}")

def parse_ts(v):
    try:
        return datetime.fromisoformat(v)
    except (TypeError, ValueError):
        return None

participants = sorted(p.name for p in EVENTS.iterdir() if p.is_dir())
print("participant folders:", len(participants))

participant folders: 50


## 1. Load once per participant and compute per-session check flags

Only per-session flags and a few numeric aggregates are retained (no raw sessions); each file is released after use.

In [2]:
read_s = parse_s = 0.0
bytes_total = 0
sessions_per_p = {}
dates_per_p = {}
flags = collections.defaultdict(list)       # check name -> list of "P###@date"
agg = collections.defaultdict(list)         # numeric aggregates
stage_names = collections.Counter()
stage_orders = collections.Counter()
key_sets = collections.Counter()
totals = collections.Counter()

def flag(name, pid, d):
    flags[name].append(f"{pid}@{d}")

for pid in participants:
    t = time.perf_counter()
    raw = (EVENTS / pid / "sleep.json").read_bytes()
    read_s += time.perf_counter() - t
    bytes_total += len(raw)
    t = time.perf_counter()
    sessions = json.loads(raw)
    parse_s += time.perf_counter() - t
    sessions_per_p[pid] = len(sessions)
    dates_per_p[pid] = collections.Counter(s.get("date") for s in sessions)
    for s in sessions:
        totals["sessions"] += 1
        d = s.get("date")
        key_sets[tuple(sorted(s))] += 1
        if s.get("participant_id") != pid:
            flag("participant_id != folder", pid, d)
        for f in ("participant_id", "date", "sleep_onset", "sleep_end", "efficiency_pct", "restlessness", "stages"):
            if s.get(f) in (None, ""):
                flag(f"null/blank {f}", pid, d)
        try:
            wake = date.fromisoformat(d)
        except (TypeError, ValueError):
            wake = None
            flag("unparseable date", pid, d)
        if wake and wake not in WINDOW_DAYS:
            flag("wake date outside study window", pid, d)
        for f in ("sleep_onset", "sleep_end"):
            if isinstance(s.get(f), str) and not TS_RE.match(s[f]):
                flag(f"{f} format != YYYY-MM-DDTHH:MM:SS.000", pid, d)
        on, end = parse_ts(s.get("sleep_onset")), parse_ts(s.get("sleep_end"))
        if on is not None and on.second:
            totals["onset with non-zero seconds"] += 1
        if on and end:
            if not on < end:
                flag("onset >= end", pid, d)
            span_min = (end - on).total_seconds() / 60
            agg["span_min"].append(span_min)
            if wake:
                if end.date() != wake:
                    flag("sleep_end date != date (wake-date misaligned)", pid, d)
                if on.date() != wake - timedelta(days=1):
                    flag("onset date != date-1 (night before)", pid, d)
                if on.date() == wake:
                    flag("onset on wake date (after midnight)", pid, d)
                agg["onset_hour"].append(on.hour + on.minute / 60)
                agg["end_hour"].append(end.hour + end.minute / 60)
        eff, rest = s.get("efficiency_pct"), s.get("restlessness")
        if isinstance(eff, (int, float)):
            agg["eff"].append(eff)
            if not 0 <= eff <= 100:
                flag("efficiency_pct outside 0-100", pid, d)
        elif eff is not None:
            flag("efficiency_pct non-numeric", pid, d)
        if isinstance(rest, (int, float)):
            agg["rest"].append(rest)
            if not 0 <= rest <= 1:
                flag("restlessness outside 0-1", pid, d)
        elif rest is not None:
            flag("restlessness non-numeric", pid, d)
        st = s.get("stages") or []
        if not isinstance(st, list) or not st:
            flag("stages empty", pid, d)
            continue
        names = [x.get("stage") for x in st]
        stage_names.update(names)
        stage_orders[tuple(names)] += 1
        if names != STAGE_ORDER:
            flag("stage order != light,deep,rem,light,awake", pid, d)
        if any(n not in {"light", "deep", "rem", "awake"} for n in names):
            flag("unknown stage name", pid, d)
        dur_sum = 0
        prev_end = None
        for x in st:
            a, b, dm = parse_ts(x.get("start_time")), parse_ts(x.get("end_time")), x.get("duration_min")
            totals["stages"] += 1
            if not isinstance(dm, int) or isinstance(dm, bool):
                flag("duration_min not an integer", pid, d)
                continue
            dur_sum += dm
            if a is None or b is None:
                flag("stage timestamp unparseable", pid, d)
                continue
            if not a < b:
                flag("stage start >= end", pid, d)
            secs = (b - a).total_seconds()
            if secs % 60:
                flag("stage span not whole minutes", pid, d)
            if round(secs / 60) != dm:
                flag("duration_min != end-start", pid, d)
                agg["duration_diff_min"].append(dm - secs / 60)
            if prev_end is not None and a != prev_end:
                flag("stage gap/overlap (start != previous end)", pid, d)
            prev_end = b
            if dm <= 0:
                flag("duration_min <= 0", pid, d)
        if on and parse_ts(st[0].get("start_time")) != on:
            flag("first stage start != sleep_onset", pid, d)
        if end and parse_ts(st[-1].get("end_time")) != end:
            flag("last stage end != sleep_end", pid, d)
        if end and parse_ts(st[-1].get("start_time")) != end:
            flag("last stage start != sleep_end", pid, d)
        if on and end:
            diff = dur_sum - (end - on).total_seconds() / 60
            agg["stage_sum_minus_span"].append(diff)
            if abs(diff) > 0.5:
                flag("sum(duration_min) != session span", pid, d)
            non_awake = sum(x["duration_min"] for x in st if x.get("stage") != "awake" and isinstance(x.get("duration_min"), int))
            agg["non_awake_minus_span"].append(non_awake - (end - on).total_seconds() / 60)
            if abs(non_awake - (end - on).total_seconds() / 60) > 0.5:
                flag("sum(non-awake duration_min) != session span", pid, d)
            if (end - on).total_seconds() > 12 * 3600:
                flag("session span > 12h", pid, d)
            if (end - on).total_seconds() < 3 * 3600:
                flag("session span < 3h", pid, d)
            asleep = sum(x["duration_min"] for x in st if x.get("stage") != "awake" and isinstance(x.get("duration_min"), int))
            if (end - on).total_seconds() > 0 and isinstance(eff, (int, float)):
                implied = 100 * asleep / ((end - on).total_seconds() / 60)
                agg["eff_minus_implied"].append(eff - implied)
                if abs(eff - implied) > 1.0:
                    flag("efficiency_pct differs >1pt from asleep/span", pid, d)
                if dur_sum > 0:
                    implied_all = 100 * asleep / dur_sum
                    agg["eff_minus_implied_all"].append(eff - implied_all)
                    if abs(eff - implied_all) > 1.0:
                        flag("efficiency_pct differs >1pt from asleep/sum(all stages)", pid, d)
    del sessions, raw

print(f"files read: {len(participants)}  sessions: {totals['sessions']}  stages: {totals['stages']}")
print(f"sleep.json read time {read_s:.3f}s, parse time {parse_s:.3f}s, bytes {bytes_total:,}, per-file mean parse {1000*parse_s/len(participants):.1f} ms")
print("session key sets:", dict(key_sets))
print("stage names:", dict(stage_names))
print("stage orders:", {"->".join(k): v for k, v in stage_orders.items()})

files read: 50  sessions: 1400  stages: 7000
sleep.json read time 0.002s, parse time 0.005s, bytes 1,489,394, per-file mean parse 0.1 ms
session key sets: {('date', 'efficiency_pct', 'participant_id', 'restlessness', 'sleep_end', 'sleep_onset', 'stages'): 1400}
stage names: {'light': 2800, 'deep': 1400, 'rem': 1400, 'awake': 1400}
stage orders: {'light->deep->rem->light->awake': 1400}


## 2. Sessions per participant vs 28 expected; duplicates

In [3]:
print("sessions per participant histogram (n->participants):", dict(sorted(collections.Counter(sessions_per_p.values()).items())))
off = {p: n for p, n in sessions_per_p.items() if n != 28}
show("participants with sessions != 28", len(off), [f"{p}={n}" for p, n in off.items()])
dup = [(p, d, c) for p, cnt in dates_per_p.items() for d, c in cnt.items() if c > 1]
show("duplicate (participant_id, date) sessions", len(dup), [f"{p}@{d}" for p, d, _ in dup])
missing = {p: sorted(d.isoformat() for d in WINDOW_DAYS if d.isoformat() not in dates_per_p[p]) for p in participants}
miss_n = sum(len(v) for v in missing.values())
show("study-window wake dates with no session (participant-days)", miss_n, [f"{p}:{v[0]}" for p, v in missing.items() if v])
print("total sessions:", totals["sessions"], "expected 50*28 =", 50 * 28)

sessions per participant histogram (n->participants): {28: 50}
participants with sessions != 28: 0
duplicate (participant_id, date) sessions: 0
study-window wake dates with no session (participant-days): 0
total sessions: 1400 expected 50*28 = 1400


## 3. Ordering, stage order, durations, whole minutes, span reconciliation, wake-date alignment, ranges

A check that is absent from the list below found nothing (count 0). Zero-count checks are listed explicitly.

In [4]:
ALL_CHECKS = [
    "participant_id != folder", "unparseable date", "wake date outside study window",
    "sleep_onset format != YYYY-MM-DDTHH:MM:SS.000", "sleep_end format != YYYY-MM-DDTHH:MM:SS.000",
    "onset >= end", "sleep_end date != date (wake-date misaligned)", "onset date != date-1 (night before)",
    "onset on wake date (after midnight)", "efficiency_pct outside 0-100", "restlessness outside 0-1",
    "efficiency_pct non-numeric", "restlessness non-numeric", "stages empty",
    "stage order != light,deep,rem,light,awake", "unknown stage name", "duration_min not an integer",
    "stage timestamp unparseable", "stage start >= end", "stage span not whole minutes",
    "duration_min != end-start", "stage gap/overlap (start != previous end)", "duration_min <= 0",
    "first stage start != sleep_onset", "last stage end != sleep_end", "last stage start != sleep_end",
    "sum(duration_min) != session span", "sum(non-awake duration_min) != session span",
    "session span > 12h", "session span < 3h",
    "efficiency_pct differs >1pt from asleep/span", "efficiency_pct differs >1pt from asleep/sum(all stages)",
]
extra = [k for k in flags if k not in ALL_CHECKS]
for name in ALL_CHECKS + sorted(extra):
    hits = flags.get(name, [])
    n_p = len({h.split("@")[0] for h in hits})
    print(f"{name}: sessions={len(hits)} participants={n_p}", sorted(hits)[:5] if hits else "(nothing found)")
print("sessions whose sleep_onset has non-zero seconds:", totals["onset with non-zero seconds"], "of", totals["sessions"])

participant_id != folder: sessions=0 participants=0 (nothing found)
unparseable date: sessions=0 participants=0 (nothing found)
wake date outside study window: sessions=0 participants=0 (nothing found)
sleep_onset format != YYYY-MM-DDTHH:MM:SS.000: sessions=0 participants=0 (nothing found)
sleep_end format != YYYY-MM-DDTHH:MM:SS.000: sessions=0 participants=0 (nothing found)
onset >= end: sessions=0 participants=0 (nothing found)
sleep_end date != date (wake-date misaligned): sessions=0 participants=0 (nothing found)
onset date != date-1 (night before): sessions=155 participants=47 ['P001@2026-02-04', 'P002@2026-01-19', 'P002@2026-01-25', 'P003@2026-01-08', 'P003@2026-01-18']
onset on wake date (after midnight): sessions=155 participants=47 ['P001@2026-02-04', 'P002@2026-01-19', 'P002@2026-01-25', 'P003@2026-01-08', 'P003@2026-01-18']
efficiency_pct outside 0-100: sessions=0 participants=0 (nothing found)
restlessness outside 0-1: sessions=0 participants=0 (nothing found)
efficiency_pc

## 4. Numeric distributions (aggregates only)

In [5]:
def summ(name, xs):
    if not xs:
        print(name, "no data"); return
    xs = sorted(xs)
    q = lambda f: xs[int(f * (len(xs) - 1))]
    print(f"{name}: n={len(xs)} min={xs[0]:.3f} p05={q(.05):.3f} median={q(.5):.3f} p95={q(.95):.3f} max={xs[-1]:.3f}")

summ("session span (min)", agg["span_min"])
summ("efficiency_pct", agg["eff"])
summ("restlessness", agg["rest"])
summ("onset local hour", agg["onset_hour"])
summ("sleep_end local hour", agg["end_hour"])
summ("sum(duration_min) minus span (min)", agg["stage_sum_minus_span"])
summ("sum(non-awake duration_min) minus span (min)", agg["non_awake_minus_span"])
summ("efficiency_pct minus asleep/span*100", agg["eff_minus_implied"])
summ("efficiency_pct minus asleep/sum(all stages)*100", agg["eff_minus_implied_all"])
print("efficiency_pct == 100.0 exactly:", sum(1 for e in agg["eff"] if e == 100.0), "of", len(agg["eff"]))
print("sessions with span > 12h:", sum(1 for x in agg["span_min"] if x > 720), " < 3h:", sum(1 for x in agg["span_min"] if x < 180))

session span (min): n=1400 min=168.000 p05=278.000 median=411.000 p95=544.000 max=1327.517
efficiency_pct: n=1400 min=50.000 p05=82.000 median=100.000 p95=100.000 max=100.000
restlessness: n=1400 min=0.026 p05=0.041 median=0.087 p95=0.160 max=0.225
onset local hour: n=1400 min=0.000 p05=0.233 median=22.950 p95=23.817 max=23.983
sleep_end local hour: n=1400 min=1.483 p05=3.400 median=5.967 p95=8.533 max=20.500
sum(duration_min) minus span (min): n=1400 min=-860.517 p05=31.000 median=59.000 p95=103.000 max=284.000
sum(non-awake duration_min) minus span (min): n=1400 min=-930.200 p05=0.000 median=0.000 p95=0.000 max=194.000
efficiency_pct minus asleep/span*100: n=1400 min=-59.845 p05=-19.000 median=0.000 p95=0.000 max=70.117
efficiency_pct minus asleep/sum(all stages)*100: n=1400 min=-39.381 p05=-2.000 median=11.857 p95=18.081 max=22.189
efficiency_pct == 100.0 exactly: 1132 of 1400
sessions with span > 12h: 3  < 3h: 2


## Runtime

In [6]:
print(f"sleep.json read {read_s:.3f}s + parse {parse_s:.3f}s; total notebook runtime: {time.perf_counter() - T0:.2f} s")

sleep.json read 0.002s + parse 0.005s; total notebook runtime: 17.85 s
